# Parking Occupancy Prediction

The goal of this project is to predict parking lot occupancy
using historical data from Lisbon parking facilities.

In [1]:
import sys
print(sys.executable)

/opt/conda/bin/python


In [2]:
import ast
import os
import numpy as np
import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)

## 1. Load and combine datasets

We load three CSV files containing historical parking occupancy.
The datetime column uses mixed date formats, so we parse it
with pandas and remove rows with missing timestamps or targets.

In [3]:
BUCKET = "s3://sagemaker-eu-north-1-527557823764/smart-parking-lot"

files = [
    f"{BUCKET}/data/raw/Lisbon_parking1.csv",
    f"{BUCKET}/data/raw/Lisbon_parking2.csv",
    f"{BUCKET}/data/raw/Lisbon_parking4.csv",
]

frames = []

for filename in files:
    part = pd.read_csv(filename, sep=";")

    part["source_file"] = filename
    part["timestamp"] = pd.to_datetime(
        part["datetime"],
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )

    frames.append(part)

df = pd.concat(frames, ignore_index=True)

df = df.dropna(
    subset=["timestamp", "occupancy", "max_capacity"]
).copy()

print("Rows after loading:", len(df))
display(df.head())

Rows after loading: 300039


,id_park,name,max_capacity,occupancy,position,datetime,entity_ts,source_file,timestamp
0,P028,Chão do Loureiro,192,0,"{'coordinates': [-9.135017, 38.712416], 'type'...",31/12/2019 23:59,2020/01/01 00:00,s3://sagemaker-eu-north-1-527557823764/smart-p...,2019-12-31 23:59:00
1,P044,Telheiras Nascente,109,25,"{'coordinates': [-9.164886, 38.761512], 'type'...",31/12/2019 23:59,2020/01/01 00:00,s3://sagemaker-eu-north-1-527557823764/smart-p...,2019-12-31 23:59:00
2,P013,Monumental,277,195,"{'coordinates': [-9.14430800000002, 38.734036]...",31/12/2019 23:59,2020/01/01 00:00,s3://sagemaker-eu-north-1-527557823764/smart-p...,2019-12-31 23:59:00
3,P004,Atrium Saldanha,251,0,"{'coordinates': [-9.1446021616448, 38.73326873...",31/12/2019 23:59,2020/01/01 00:01,s3://sagemaker-eu-north-1-527557823764/smart-p...,2019-12-31 23:59:00
4,P040,Mercado de Alvalade,118,88,"{'coordinates': [-9.139498, 38.755424], 'type'...",01/01/2020 00:03,2020/01/01 00:05,s3://sagemaker-eu-north-1-527557823764/smart-p...,2020-01-01 00:03:00


In [4]:
file_counts = df.groupby("id_park")["source_file"].nunique()

selected_ids = file_counts[file_counts == 3].index

df = df[df["id_park"].isin(selected_ids)].copy()

print("Number of selected parking lots:", len(selected_ids))
print("Selected IDs:", sorted(selected_ids))
print("Rows after filtering:", len(df))

Number of selected parking lots: 21
Selected IDs: ['P002', 'P004', 'P006', 'P008', 'P015', 'P017', 'P019', 'P020', 'P024', 'P028', 'P031', 'P033', 'P035', 'P037', 'P040', 'P042', 'P044', 'P046', 'P048', 'P057', 'P062']
Rows after filtering: 283284


In [5]:
rows_before = len(df)

df = df.drop_duplicates(
    subset=["id_park", "timestamp", "occupancy"]
).copy()

print("Rows before deduplication:", rows_before)
print("Rows after deduplication:", len(df))
print("Duplicates removed:", rows_before - len(df))

Rows before deduplication: 283284
Rows after deduplication: 282711
Duplicates removed: 573


In [6]:
def extract_coordinates(value):
    try:
        coordinates = ast.literal_eval(value)["coordinates"]
        return coordinates[0], coordinates[1]
    except (ValueError, SyntaxError, TypeError, KeyError):
        return np.nan, np.nan


df[["lon", "lat"]] = df["position"].apply(
    lambda value: pd.Series(extract_coordinates(value))
)

rows_before = len(df)
df = df.dropna(subset=["lon", "lat"]).copy()

print("Rows removed due to invalid coordinates:", rows_before - len(df))
display(df[["id_park", "position", "lon", "lat"]].head())

Rows removed due to invalid coordinates: 0


,id_park,position,lon,lat
0,P028,"{'coordinates': [-9.135017, 38.712416], 'type'...",-9.135017,38.712416
1,P044,"{'coordinates': [-9.164886, 38.761512], 'type'...",-9.164886,38.761512
3,P004,"{'coordinates': [-9.1446021616448, 38.73326873...",-9.144602,38.733269
4,P040,"{'coordinates': [-9.139498, 38.755424], 'type'...",-9.139498,38.755424
5,P002,"{'coordinates': [-9.1483930362, 38.7313746], '...",-9.148393,38.731375


In [7]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

display(
    df[
        ["timestamp", "hour", "day_of_week", "month", "is_weekend"]
    ].head()
)

,timestamp,hour,day_of_week,month,is_weekend
0,2019-12-31 23:59:00,23,1,12,0
1,2019-12-31 23:59:00,23,1,12,0
3,2019-12-31 23:59:00,23,1,12,0
4,2020-01-01 00:03:00,0,2,1,0
5,2020-01-01 00:04:00,0,2,1,0


In [8]:
FEATURES = [
    "max_capacity",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lon",
    "lat",
]

TARGET = "occupancy"

print("Features:", FEATURES)
print("Target:", TARGET)

display(df[FEATURES + [TARGET]].head())

Features: ['max_capacity', 'hour', 'day_of_week', 'month', 'is_weekend', 'lon', 'lat']
Target: occupancy


,max_capacity,hour,day_of_week,month,is_weekend,lon,lat,occupancy
0,192,23,1,12,0,-9.135017,38.712416,0
1,109,23,1,12,0,-9.164886,38.761512,25
3,251,23,1,12,0,-9.144602,38.733269,0
4,118,0,2,1,0,-9.139498,38.755424,88
5,400,0,2,1,0,-9.148393,38.731375,275


In [9]:
timestamps = np.sort(df["timestamp"].unique())

train_index = int(len(timestamps) * 0.70)
val_index = int(len(timestamps) * 0.85)

train_end = timestamps[train_index]
val_end = timestamps[val_index]

train = df[df["timestamp"] < train_end].copy()

validation = df[
    (df["timestamp"] >= train_end)
    & (df["timestamp"] < val_end)
].copy()

test = df[df["timestamp"] >= val_end].copy()

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = validation[FEATURES]
y_val = validation[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print("Train:", len(train), train["timestamp"].min(), "->", train["timestamp"].max())
print("Validation:", len(validation), validation["timestamp"].min(), "->", validation["timestamp"].max())
print("Test:", len(test), test["timestamp"].min(), "->", test["timestamp"].max())

Train: 200554 2019-12-31 23:59:00 -> 2020-10-26 11:22:00
Validation: 45683 2020-10-26 11:24:00 -> 2020-11-27 22:24:00
Test: 36474 2020-11-27 22:28:00 -> 2020-12-31 23:54:00


In [10]:
TRACKING_URI = "sqlite:////home/sagemaker-user/smart-parking-lot/mlflow.db"
MODEL_NAME = "parking-occupancy-model"
EXPERIMENT = "parking-occupancy-prediction"

mlflow.set_tracking_uri(TRACKING_URI)

if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(
        EXPERIMENT,
        artifact_location=f"{BUCKET}/mlflow-artifacts",
    )
mlflow.set_experiment(EXPERIMENT)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Artifacts:", mlflow.get_experiment_by_name(EXPERIMENT).artifact_location)


MLflow tracking URI: sqlite:////home/sagemaker-user/smart-parking-lot/mlflow.db
Artifacts: s3://sagemaker-eu-north-1-527557823764/smart-parking-lot/mlflow-artifacts


In [11]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 300, step=100
        ),
        "max_depth": trial.suggest_categorical(
            "max_depth", [10, 15, 20, None]
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 8
        ),
        "max_features": trial.suggest_float(
            "max_features", 0.6, 1.0
        ),
    }

    with mlflow.start_run(
        run_name=f"optuna_trial_{trial.number}",
        nested=True,
    ):
        mlflow.log_params(params)

        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        val_predictions = model.predict(X_val)

        val_mae = mean_absolute_error(
            y_val,
            val_predictions,
        )

        mlflow.log_metric("validation_mae", val_mae)

        return val_mae

In [12]:
study = optuna.create_study(
    direction="minimize",
    study_name="random-forest-optimization",
)

with mlflow.start_run(run_name="optuna_search"):
    study.optimize(
        objective,
        n_trials=10,
        show_progress_bar=True,
    )

    mlflow.log_params(
        {f"best_{k}": v for k, v in study.best_params.items()}
    )
    mlflow.log_metric("best_validation_mae", study.best_value)

print("Best validation MAE:", round(study.best_value, 2))
print("Best parameters:")
print(study.best_params)


[I 2026-10-25 13:49:03,090] A new study created in memory with name: random-forest-optimization


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-10-25 13:49:14,776] Trial 0 finished with value: 10.359325481629787 and parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 0.7822375367625671}. Best is trial 0 with value: 10.359325481629787.
[I 2026-10-25 13:49:28,332] Trial 1 finished with value: 10.332895189214408 and parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 0.935216190304637}. Best is trial 1 with value: 10.332895189214408.
[I 2026-10-25 13:49:56,077] Trial 2 finished with value: 10.994429070464927 and parameters: {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 0.6554733540669953}. Best is trial 1 with value: 10.332895189214408.
[I 2026-10-25 13:50:08,393] Trial 3 finished with value: 9.679457231684088 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 18, 'min_samples_leaf': 5, 'max_features': 0.657412537111856

In [13]:
trials_df = study.trials_dataframe()
display(
    trials_df.sort_values("value")[
        ["number", "value", "params_n_estimators",
         "params_max_depth", "params_min_samples_split",
         "params_min_samples_leaf", "params_max_features"]
    ]
)

,number,value,params_n_estimators,params_max_depth,params_min_samples_split,params_min_samples_leaf,params_max_features
9,9,9.645049,100,15.0,6,6,0.622924
3,3,9.679457,100,20.0,18,5,0.657413
5,5,9.705038,200,15.0,17,3,0.775402
8,8,9.720218,200,NaN,2,7,0.777669
4,4,9.723528,300,15.0,19,6,0.884624
6,6,9.776680,100,NaN,4,7,0.877856
7,7,10.020957,200,20.0,2,3,0.880466
1,1,10.332895,100,10.0,7,4,0.935216
0,0,10.359325,100,10.0,12,3,0.782238
2,2,10.994429,300,10.0,5,3,0.655473


In [14]:
train_full = pd.concat([train, validation]).sort_values("timestamp")

X_train_full = train_full[FEATURES]
y_train_full = train_full[TARGET]

best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
)

best_model.fit(X_train_full, y_train_full)

test_predictions = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print("=== FINAL TEST RESULTS ===")
print("MAE:", round(test_mae, 2))
print("RMSE:", round(test_rmse, 2))
print("R²:", round(test_r2, 4))

=== FINAL TEST RESULTS ===
MAE: 10.05
RMSE: 19.5
R²: 0.9914


In [15]:
# Calculate each parking lot's mean occupancy using training data only
lot_means = train.groupby("id_park")["occupancy"].mean()

# Predict test occupancy using the training-period mean
baseline_predictions = (
    test["id_park"]
    .map(lot_means)
    .fillna(train["occupancy"].mean())
)

# Evaluate baseline
baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)
baseline_r2 = r2_score(y_test, baseline_predictions)

print("=== BASELINE: PARKING LOT MEAN ===")
print("MAE:", round(baseline_mae, 2))
print("RMSE:", round(baseline_rmse, 2))
print("R²:", round(baseline_r2, 4))

# Compare against the final Optuna-tuned model
print("\n=== OPTUNA MODEL vs BASELINE ===")
print("Optuna model MAE:", round(test_mae, 2))
print("Baseline MAE:", round(baseline_mae, 2))

if baseline_mae > 0:
    improvement = (
        (baseline_mae - test_mae) / baseline_mae * 100
    )
    print("MAE improvement:", round(improvement, 2), "%")

=== BASELINE: PARKING LOT MEAN ===
MAE: 27.35
RMSE: 46.59
R²: 0.951

=== OPTUNA MODEL vs BASELINE ===
Optuna model MAE: 10.05
Baseline MAE: 27.35
MAE improvement: 63.25 %


In [16]:
with mlflow.start_run(run_name="final_random_forest"):
    mlflow.log_params(study.best_params)
    mlflow.log_param("features", ",".join(FEATURES))

    mlflow.log_metrics({
        "best_validation_mae": float(study.best_value),
        "test_mae": float(test_mae),
        "test_rmse": float(test_rmse),
        "test_r2": float(test_r2),
        "baseline_mae": float(baseline_mae),
        "baseline_rmse": float(baseline_rmse),
        "baseline_r2": float(baseline_r2),
    })

    sample = X_train_full.head(100)
    signature = infer_signature(sample, best_model.predict(sample))

    # Logs the model AND registers it in the Model Registry
    # (creates version 1, 2, 3... of MODEL_NAME on every run).
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        signature=signature,
        input_example=sample.head(5),
        registered_model_name=MODEL_NAME,
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

# Point the "champion" alias at the newest version.
# (Fine for a solo project; later you can promote only if test_mae improves.)
client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)
client.set_registered_model_alias(MODEL_NAME, "champion", latest_version)

print(f"Registered {MODEL_NAME} v{latest_version} -> alias 'champion'")

# Sanity check: load it back from the registry
loaded = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
print("Loaded model predictions:", loaded.predict(X_test.head(5)))


/opt/conda/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered parking-occupancy-model v1 -> alias 'champion'


Successfully registered model 'parking-occupancy-model'.
Created version '1' of model 'parking-occupancy-model'.


Loaded model predictions: [ 43.86916978  96.67410378  25.21944434  56.74285672 330.15290438]


In [17]:
from pathlib import Path

parking_lots = (
    df.sort_values("timestamp")
    .groupby("id_park", as_index=False)
    .agg(
        max_capacity=("max_capacity", "first"),
        lon=("lon", "first"),
        lat=("lat", "first"),
    )
)

parking_lots.to_csv(f"{BUCKET}/data/parking_lots.csv", index=False)

print(f"Saved {len(parking_lots)} parking lots")
display(parking_lots.head())

Saved 21 parking lots


,id_park,max_capacity,lon,lat
0,P002,400,-9.148393,38.731375
1,P004,251,-9.144602,38.733269
2,P006,255,-9.142094,38.737444
3,P008,513,-9.151761,38.736616
4,P015,620,-9.155042,38.750596
